In [42]:
# Import Libraries and Load Cleaned Data
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path("D:/Datasets/Research/Customer Churn/Telco Customer Churn/telco_cleaned.csv"))
df.shape

(7032, 22)

Basic Target and ID handling

Drop customerID as it is an identifier, not a predictive feature

In [43]:
# Drop the identifier 'customerID' and index column 'Unnamed: 0'
df = df.drop(columns = ['customerID', 'Unnamed: 0'])


In [44]:
# Convert the Churn feature to integer
df["Churn"] = df["Churn"].map({"Yes":1, "No":0}).fillna(df["Churn"]).astype(int)

 Fix "Yes/No" columns into 0/1 for cleaner modelling

In [45]:
yes_no_cols = [
    "Partner", "Dependents", "PhoneService", "PaperlessBilling", "MultipleLines", "OnlineSecurity", 
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]

for col in yes_no_cols:
    if col in df.columns:
        df[col] = df[col].replace({"Yes":1, "No":0})

C:\Users\natha\AppData\Local\Temp\ipykernel_26472\3912218853.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({"Yes":1, "No":0})


### Collapse special categories (No internet/phone service)

#### Internet-related columns

For these columns:

OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies

Treat:

- "No internet service" as 0

- "Yes" as 1
- "No" as 0

In [46]:
internet_service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]

for col in internet_service_cols:
    if col in df.columns:
        df[col] = df[col].replace({"No internet service":0, "No":0, "Yes":1})

C:\Users\natha\AppData\Local\Temp\ipykernel_26472\3403547689.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({"No internet service":0, "No":0, "Yes":1})


In [47]:
# For MultipleLines column, treat "No phone service" as 0.

if "MultipleLines" in df.columns:
    df["MultipleLines"] = df["MultipleLines"].replace({"No phone service":0, "No":0, "Yes":1})

C:\Users\natha\AppData\Local\Temp\ipykernel_26472\3032450542.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["MultipleLines"] = df["MultipleLines"].replace({"No phone service":0, "No":0, "Yes":1})


### Create tenure buckets 

In [48]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins = [-1, 12, 24, 48, 72],
    labels = ["0-12", "13-24", "25-48", "49-72"]
)

### Create spend-based features

In [49]:
df["avg_monthly_spend"] = (df["TotalCharges"] / df["tenure"]).replace([float("inf")], 0).fillna(0)
df["high_monthly_charges"] = (df["MonthlyCharges"] > df["MonthlyCharges"].median()).astype(int)

In [50]:
df["charges_per_tenure_bucket"] = df["MonthlyCharges"] * (df["tenure"] / (df["tenure"].max()))

### One-hot encode remaining categorical variables

In [51]:
# Encode only the object columns
cat_cols = df.select_dtypes(include = "object").columns.tolist()
cat_cols

['gender', 'InternetService', 'Contract', 'PaymentMethod']

In [52]:
df_model = pd.get_dummies(df, columns = cat_cols, drop_first = True)
df_model.shape

(7032, 28)

In [53]:
df_model = pd.get_dummies(df_model, columns = ["tenure_group"], drop_first=True)

### Split X/y and save processed dataset

In [54]:
# Split the data
X = df_model.drop(columns = ["Churn"])
y = df_model["Churn"]

In [55]:
# Save the processed data
Path("data/processed").mkdir(parents = True, exist_ok = True)

X.to_csv("data/processed/X_features.csv", index = False)
y.to_csv("data/processed/y_target.csv", index = False)

Quick Sanity Checks

In [56]:
print("Any missing values in X?", X.isna().any().any())
print("Target distribution:\n", y.value_counts(normalize=True))
print("X shape:", X.shape)


Any missing values in X? False
Target distribution:
 Churn
0    0.734215
1    0.265785
Name: proportion, dtype: float64
X shape: (7032, 29)


## Feature Engineering Summary

- Converted service flags (Yes/No, No internet service) into binary indicators

- Created tenure groups for interpretability

- Engineered spend features (avg_monthly_spend, high_monthly_charges)

- One-hot encoded remaining categorical variables

- Saved modelling-ready X and y to /data/processed